In [1]:
# Import all the Libraries (Code Snippet from Lecture 4)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
import re
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# 1.1 Load and Inspect the Data

In [2]:
# Load and Display the Data with SQL
with sqlite3.connect("data.db") as conn:
    df = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

# First Few Rows
df.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [3]:
# Display Basic Statistics
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [4]:
# Identify the column names and meanings
print("Columns in dataset:", df.columns.tolist())

Columns in dataset: ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']


Column name meanings:

ts means timestamp, representing the date for each row (ex. 1975-04-14)

open means the opening price at the start of the trading period (ex. 5.4300)

high means the highest price point reached during the period, (ex. 5.4800)

low means the lowest price point reached during the period, (ex. 5.3600)

close is the closing price at the end of the period (ex. 5.4400)

adj_close means the adjusted closing price after adjusting for dividends or stock splits (ex. 1.640000)

volume is the amount of shares traded within the given period (ex. 446392)

ticker is the stock symbol for the asset (ex. CAT)

In [5]:
# Show the date range
df['ts'] = pd.to_datetime(df['ts'])
min_date = df['ts'].min().strftime('%Y-%m-%d')
max_date = df['ts'].max().strftime('%Y-%m-%d')
print(f"Date Range: {min_date} to {max_date}")

# NOTE: there are a lot of raw HTML strings in the ticker column before 
# data cleaning, so lets clean that
df['ticker'] = (
    df['ticker'].astype(str).str.replace(r'<[^>]*>', '', regex=True).str.strip()
)

# and list of tickers
tickers = df['ticker'].unique().tolist()
print(f"Tickers ({len(tickers)} total): {tickers}")

Date Range: 1962-01-02 to 2024-12-06
Tickers (30 total): ['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ', 'DIS', 'CSCO', 'SHW', 'KO', 'VZ', 'IBM', 'PG', 'HON', 'AXP', 'MMM', 'GS', 'BA', 'CRM', 'UNH', 'NKE', 'NVDA', 'AMGN', 'MCD', 'V', 'MRK', 'WMT', 'JPM', 'MSFT', 'AMZN']


# 1.2 Detect Data Quality Issues

In [6]:
# Initialize issues list to store them in a table
issues = []

In [7]:
# Identify missing values by column
print('--- Missing Values Count per Column ---')
print(df.isnull().sum())

# Add the Missing values to the issues table
for col in ['open', 'high', 'low', 'close', 'adj_close', 'volume']:
  for idx in df[df[col].isna()].index:
    issues.append({
        'ticker': df.loc[idx, 'ticker'],
        'issue_type': 'Missing Value (NaN)',
        'issue_date': df.loc[idx, 'ts'],
        'description': f"Column '{col}' contains a missing value.",
        'severity': 'high',
    })

--- Missing Values Count per Column ---
ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64


In [8]:
# Detect missing trader days (gaps in the time series)
for ticker, group in df.groupby('ticker'):
  start, end = group['ts'].min(), group['ts'].max()
  missing = pd.bdate_range(start, end).difference(
      pd.DatetimeIndex(group['ts'].dt.normalize())
  )

  print(
      f"Ticker {ticker} is missing {len(missing)} trading days between"
      f" {start:%Y-%m-%d} and {end:%Y-%m-%d}"
  )

  # Append missing days to the issues report
  for m_date in missing:
    issues.append({
        'ticker': ticker,
        'issue_type': 'Missing Trading Day',
        'issue_date': m_date,
        'description': f'Trading day {m_date.strftime("%Y-%m-%d")} is missing.',
        'severity': 'medium',
    })

Ticker AAPL is missing 391 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 365 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 255 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 482 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 600 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 600 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 186 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 312 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 600 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 576 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 236 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 378 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 600 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 600 trading days between 196

In [9]:
# Ensure there are no duplicated rows
df = df.sort_values(['ticker', 'ts']).drop_duplicates(subset=['ticker', 'ts'], keep='first').reset_index(drop=True)

# Calculate Daily Returns on Close and Adjusted Close
# I added the adj_returns because I think its a useful statistic, and I will use it in the outliers portion (cell 10)
df['prev_close'] = df.groupby('ticker')['close'].shift(1)
df['return'] = (df['close'] - df['prev_close']) / df['prev_close']

df['prev_adj_close'] = df.groupby('ticker')['adj_close'].shift(1)
df['adj_return'] = (df['adj_close'] - df['prev_adj_close']) / df['prev_adj_close']

In [10]:
# Flag abs(return - mean_return) > 3 * std_return or extreme absolute returns for each tail within each ticker

# Mean and std
mean_ret = df.groupby('ticker')['return'].transform('mean')
std_ret = df.groupby('ticker')['return'].transform(lambda x: x.std(ddof=1))

# Mean and std for the adj return 
mean_adj_ret = df.groupby('ticker')['adj_return'].transform('mean')
std_adj_ret = df.groupby('ticker')['adj_return'].transform(lambda x: x.std(ddof=1))

# Initialize flag column
df['return_outlier_flag'] = False

# Check for valid standard deviation and non-null returns
valid_mask = std_ret.notna() & (std_ret > 0) & df['return'].notna()
outlier_close = valid_mask & (abs(df['return'] - mean_ret) > 3 * std_ret)

valid_adj_mask = std_adj_ret.notna() & (std_adj_ret > 0) & df['adj_return'].notna()
outlier_adj = valid_adj_mask & (abs(df['adj_return'] - mean_adj_ret) > 3 * std_adj_ret)

# Flag if EITHER raw close return or adjusted close return exceeds 3 sigma
# Added the adj_close because when I went to go look at the AAPL data for notebook 2
# there was messy data that I hadn't flagged appearing in my plot, adding the 
# scanning for adj_close jumps fixes this
outlier_mask = outlier_close | outlier_adj
df.loc[outlier_mask, 'return_outlier_flag'] = True

print(f"Total return outliers flagged for review: {df['return_outlier_flag'].sum()}")

# Append return outliers to the issues table
for idx in df[df['return_outlier_flag']].index:
    bad_ret = df.loc[idx, 'adj_return'] if outlier_adj.loc[idx] else df.loc[idx, 'return']
    issues.append({
        'ticker': df.loc[idx, 'ticker'],
        'issue_type': 'Return Outlier (>3σ)',
        'issue_date': df.loc[idx, 'ts'],
        'description': f"Daily return {bad_ret:.4f} exceeds 3 sigma threshold.",
        'severity': 'medium',
    })

Total return outliers flagged for review: 12442


In [11]:
# Flag any day where: high < close or low > close
df['impossible_bound_flag'] = (df['high'] < df['close']) | (
    df['low'] > df['close']
)

print(
    'Total rows with impossible price bounds:'
    f" {df['impossible_bound_flag'].sum()}"
)

# YAY THE DATA DOES NOT HAVE ANY IMPOSSIBLE BOUND ERRORS, 

Total rows with impossible price bounds: 0


In [12]:
# Flag nonpositive or non-finite equity prices, including adjusted close; 
# raw OHLC bounds do not apply to adjusted close
price_cols = ['open', 'high', 'low', 'close', 'adj_close']

# Initialize flag column
df['invalid_price_flag'] = False

for col in price_cols:
  # Flag if the value is not null and is negative or infinite
  invalid_mask = df[col].notna() & ((df[col] <= 0) | (~np.isfinite(df[col])))
  df.loc[invalid_mask, 'invalid_price_flag'] = True

  # Append invalid prices to the issues table
  for idx in df[invalid_mask].index:
    issues.append({
        'ticker': df.loc[idx, 'ticker'],
        'issue_type': 'Invalid Price',
        'issue_date': df.loc[idx, 'ts'],
        'description': f"Column '{col}' has invalid value: {df.loc[idx, col]}.",
        'severity': 'high',
    })

print(
    'Total rows with nonpositive or non-finite equity prices:'
    f" {df['invalid_price_flag'].sum()}"
)

Total rows with nonpositive or non-finite equity prices: 5


In [13]:
# Inconsistencies Listed:
# Verify that low <= open <= high and low <= close <= high
df['ohlc_inconsistent_flag'] = (
    (df['open'] < df['low'])
    | (df['open'] > df['high'])
    | (df['close'] < df['low'])
    | (df['close'] > df['high'])
)

# Check for zero volume days (potential stale data indicator)
df['zero_volume_flag'] = df['volume'] == 0

print(f"Inconsistent OHLC range rows: {df['ohlc_inconsistent_flag'].sum()}")
print(f"Zero volume rows: {df['zero_volume_flag'].sum()}")

# Append inconsistent OHLC ranges to issues list
for idx in df[df['ohlc_inconsistent_flag']].index:
  issues.append({
      'ticker': df.loc[idx, 'ticker'],
      'issue_type': 'Inconsistent OHLC Range',
      'issue_date': df.loc[idx, 'ts'],
      'description': (
          f"Open ({df.loc[idx, 'open']}) or Close ({df.loc[idx, 'close']})"
          f" violates [Low ({df.loc[idx, 'low']}), High"
          f" ({df.loc[idx, 'high']})] bounds."
      ),
      'severity': 'high',
  })

# Append zero volume days to issues list
for idx in df[df['zero_volume_flag']].index:
  issues.append({
      'ticker': df.loc[idx, 'ticker'],
      'issue_type': 'Zero Volume',
      'issue_date': df.loc[idx, 'ts'],
      'description': 'Trading volume is zero (potential stale data).',
      'severity': 'low',
  })

Inconsistent OHLC range rows: 5
Zero volume rows: 75


In [14]:
# Create the issues table
if 'issues' in locals() and issues:
  dq_report_df = pd.DataFrame(issues)[
      ['ticker', 'issue_type', 'issue_date', 'description', 'severity']
  ]
else:
  dq_report_df = pd.DataFrame(
      columns=['ticker', 'issue_type', 'issue_date', 'description', 'severity']
  )

print(f'Total issues recorded in Data Quality Report: {len(dq_report_df)}')
display(dq_report_df.head(20))

Total issues recorded in Data Quality Report: 45984


,ticker,issue_type,issue_date,description,severity
0,KO,Missing Value (NaN),2024-04-16,Column 'adj_close' contains a missing value.,high
1,NVDA,Missing Value (NaN),2018-06-12,Column 'adj_close' contains a missing value.,high
2,HD,Missing Value (NaN),2013-04-16,Column 'adj_close' contains a missing value.,high
3,PG,Missing Value (NaN),2015-03-20,Column 'adj_close' contains a missing value.,high
4,JPM,Missing Value (NaN),2018-03-13,Column 'adj_close' contains a missing value.,high
5,NVDA,Missing Value (NaN),2000-05-15,Column 'adj_close' contains a missing value.,high
6,CSCO,Missing Value (NaN),2001-10-02,Column 'adj_close' contains a missing value.,high
7,MMM,Missing Value (NaN),1975-07-03,Column 'adj_close' contains a missing value.,high
8,IBM,Missing Value (NaN),2019-06-21,Column 'adj_close' contains a missing value.,high
9,CSCO,Missing Value (NaN),2006-01-09,Column 'adj_close' contains a missing value.,high


##Explanation of how severity was chosen:
1) high: Assigned to data that breaks the rules of financial logic and could break a model. Such as non-positive or non-finite prices, prices violating basic OHLC bounds like high < close or low > close, and Open/Close falling outside [Low, High]).
2) moderate: statistical abnormalities like lying outside the standard deviation range, or missing trading days
3) low: Assigned to zero volume days, which may indicate stale data or halted trading rather than an explicit issue in the data

In [15]:
# Pre-repair diagnosis report
dq_report_before = dq_report_df.copy()
print(f"Pre-repair baseline established with {len(dq_report_before)} issues.")
display(dq_report_before['issue_type'].value_counts())

Pre-repair baseline established with 45984 issues.


issue_type
Missing Value (NaN)        19855
Missing Trading Day        13602
Return Outlier (>3σ)       12442
Zero Volume                   75
Invalid Price                  5
Inconsistent OHLC Range        5
Name: count, dtype: int64

In [16]:
# Demonstrate the detector and repair behavior on small injected examples for required cases absent from database.

# Slice a section of data for demonstration
demo_df = df.head(3).copy()

# Create some issues that we can detect
demo_df.loc[0, 'high'] = demo_df.loc[0, 'low'] - 2.0  # Impossible bound: High < Low
demo_df.loc[1, 'close'] = -5.0                         # Invalid price: Negative close
demo_df.loc[2, 'volume'] = 0                           # Zero volume

# Detect issues on the demo slice
demo_issues = []
for idx, row in demo_df.iterrows():
  if row['high'] < row['low']:
    demo_issues.append({'ticker': row['ticker'], 'issue_type': 'Impossible Bound', 'issue_date': row['ts'], 'severity': 'high'})
  if row['close'] <= 0:
    demo_issues.append({'ticker': row['ticker'], 'issue_type': 'Invalid Price', 'issue_date': row['ts'], 'severity': 'high'})
  if row['volume'] == 0:
    demo_issues.append({'ticker': row['ticker'], 'issue_type': 'Zero Volume', 'issue_date': row['ts'], 'severity': 'low'})

demo_report = pd.DataFrame(demo_issues)
display(demo_report)

# Repair the slice to demonstrate 
demo_df.loc[0, 'high'] = demo_df.loc[0, 'low'] + 5.0     # Repair bound
demo_df.loc[1, 'close'] = abs(demo_df.loc[1, 'close'])    # Fix negative price
demo_df.loc[2, 'volume'] = 1250000                        # Impute baseline volume

# Verify no issues post repair
post_repair_issues = []
for idx, row in demo_df.iterrows():
  if row['high'] < row['low'] or row['close'] <= 0 or row['volume'] == 0:
    post_repair_issues.append(row['ticker'])

print(f"Remaining issues in demo slice after repair: {len(post_repair_issues)}")

,ticker,issue_type,issue_date,severity
0,AAPL,Impossible Bound,1980-12-12,high
1,AAPL,Invalid Price,1980-12-15,high
2,AAPL,Zero Volume,1980-12-16,low


Remaining issues in demo slice after repair: 0


# 1.3 Clean the Data

In [17]:
# Functions to handle missing values
def handle_missing_values(df):
    df = df.copy()
    df = df.sort_values(['ticker', 'ts']).reset_index(drop=True)
    
    # Initialize quarantine status column
    if 'quarantine_status' not in df.columns:
        df['quarantine_status'] = 'normal'
    
    raw_price_cols = ['open', 'high', 'low', 'close']
    
    # Drop rows where all raw prices are missing before filling
    all_missing_mask = df[raw_price_cols].isna().all(axis=1)
    quarantined_all_missing = df[all_missing_mask].copy()
    df = df[~all_missing_mask].copy()
    
    # Handle missing volume: Keep as zero and ensure repair flag is not removed
    vol_missing_mask = df['volume'].isna()
    df.loc[vol_missing_mask, 'volume'] = 0
    
    # Handle missing prices: Use forward-fill (carry last known price)
    df[raw_price_cols] = df.groupby('ticker')[raw_price_cols].ffill()

    # Handle missing adj_close using trusted adjustment factors
    """
    For this part, if the adjusted close is missing but the close is present, we can
    use other rows where the adjusted close and raw close are present to make a 
    rescaling factor. (This is a logical correction to make because the adjusted close
    represents the close adjusted for stock splits and dividends.) It does not make sense
    to forward fill because stock splits having a large impact on the values. For this we
    will include a limit to ensure no forward filling in large time gaps. Finally, if 
    there is no factor to adjust by, quarantine that row to ensure tha data remains valid.
    """
    # Calculate the Adjustment Factor (for rows where the close and adj_close are present)
    corrupted_adj_mask = df['adj_close'].notna() & df['close'].notna() & (df['adj_close'] > df['close'] * 1.5)
    df.loc[corrupted_adj_mask, 'adj_close'] = np.nan
    df['adj_factor'] = np.nan
    valid_factor_mask = (
        df['close'].notna()
        & (df['close'] > 0)
        & df['adj_close'].notna()
        & (df['adj_close'] > 0)
    )
    df.loc[valid_factor_mask, 'adj_factor'] = (
        df.loc[valid_factor_mask, 'adj_close'] / df.loc[valid_factor_mask, 'close']
    )
    
    # Limit forward-fill with adjustment factor to time gaps less than 5 trading days 
    # Longest trading gap I can think of is Christmas Eve and Day + 2 weekend days, so 4 days
    df['adj_factor'] = df.groupby('ticker')['adj_factor'].ffill(limit=5)
    
    # Mark rows where adj_close is missing and no reliable adjustment factor exists
    unresolved_adj_mask = df['adj_close'].isna() & df['adj_factor'].isna()
    df.loc[unresolved_adj_mask, 'quarantine_status'] = 'unresolved_adj_close'
    
    # Reconstruct adj_close from validated raw close and factor
    reconstruct_mask = df['adj_close'].isna() & df['adj_factor'].notna()
    df.loc[reconstruct_mask, 'adj_close'] = (
        df.loc[reconstruct_mask, 'close'] * df.loc[reconstruct_mask, 'adj_factor']
    )
    
    # Drop temporary adjustment factor column
    df = df.drop(columns=['adj_factor'])
    return df

In [18]:
# Funtions to Handle Outliers (Impossible price relationships eg. high < close)
def fix_impossible_prices_hybrid(df: pd.DataFrame) -> pd.DataFrame:
  """I decided to use a hybrid approach:

  1. First determine which value seems incorrect, for example if the high is
  less than all the OHLC,
  then we will change the high value. If for example the close is less than
  the low, then the close
  can be replaced. (ie, use context to determine which to correct)
  2. Begin by using option 1 (forward filling the chosen data piece)
  3. If the ffill fails, for example forward filling the close results in a
  value greater than the high,
  use a secondary method of option 3, which is to extrapolate using today's
  data.
  """
  df_clean = df.copy()
  df_clean = df_clean.sort_values(by=["ticker", "ts"]).reset_index(drop=True)
  price_cols = ["open", "high", "low", "close", "adj_close"]

  # PASS 1: Identify and flag contextual outliers as NaN
  for idx, row in df_clean.iterrows():
    o, h, l, c = row["open"], row["high"], row["low"], row["close"]

    # Flag zero or negative values across all price columns
    for col in price_cols:
      if row[col] <= 0:
        df_clean.at[idx, col] = np.nan

    # Inverted High/Low range
    if h < l:
      df_clean.at[idx, "high"] = np.nan
      df_clean.at[idx, "low"] = np.nan
      continue

    # Invalid High relationship
    if h < c or h < o:
      if c > 0 and (c - o) > (o - h):
        df_clean.at[idx, "close"] = np.nan
      else:
        df_clean.at[idx, "high"] = np.nan

    # Invalid Low relationship
    if l > c or l > o:
      if c > 0 and (o - c) > (l - o):
        df_clean.at[idx, "close"] = np.nan
      else:
        df_clean.at[idx, "low"] = np.nan

  # PASS 2: Option 1 Grouped Forward-Fill & Back-Fill
  df_clean[price_cols] = df_clean.groupby("ticker")[price_cols].ffill().bfill()

  # PASS 3: Option 3 Fallback Row Interpolation for persistent invalid bounds
  still_bad_high = (
      (df_clean["high"] < df_clean["open"])
      | (df_clean["high"] < df_clean["close"])
      | (df_clean["high"] < df_clean["low"])
  )
  still_bad_low = (
      (df_clean["low"] > df_clean["open"])
      | (df_clean["low"] > df_clean["close"])
      | (df_clean["low"] > df_clean["high"])
  )
  still_bad_close = (df_clean["close"] > df_clean["high"]) | (
      df_clean["close"] < df_clean["low"]
  )

  # Fallback Option 3 logic
  df_clean.loc[still_bad_high, "high"] = df_clean.loc[
      still_bad_high, ["open", "close", "low"]
  ].max(axis=1)
  df_clean.loc[still_bad_low, "low"] = df_clean.loc[
      still_bad_low, ["open", "close", "high"]
  ].min(axis=1)
  df_clean.loc[still_bad_close, "close"] = (
      df_clean.loc[still_bad_close, "high"] + df_clean.loc[still_bad_close, "low"]
  ) / 2
    
  return df_clean

### Explained choice of how to handle impossible price relationships

Selected Approach: Hybrid Method (Option 1 Forward-Fill + Option 3 Midpoint Fallback)**

* **Primary Method (Option 1):** Use the context to identify the invalid pries, set them to 'NaN', and forward fill within each ticker to preserve continuity. 
* **Fallback Safety Net (Option 3):** If a forward-filled price still violates daily boundary conditions (e.g., due to large gap-ups/downs across trading sessions or trading days), recalculate the value using same-row boundary bounds ($High = \max(O, C)$, $Close = \frac{H + L}{2}$).

Note: slight caviat of chosen method, relies on the open being correct and trustworthy. I had a discussion with Gemini about exploring other options that were less biased than the open, but decided I didn't love the other options. One of the suggested options was to take the median of the OHLC and determine which value seems invalid, but I think that most cases of this involve taking the average of the open and close, leading to a similar bias, and an overbias of adjusting the high and close (since they are more likely to be outliers). 

In [19]:
# Functions to Handle Outliers (For extreme returns >3 std dev)
def process_flagged_outliers(
    df: pd.DataFrame, outlier_indices: pd.Index
) -> pd.DataFrame:
    """Classify >3 std dev errors as either:
    - Data Errors that should be interpolated
    - Valid Events: Stock splits, earnings, or real market shocks (don't change and keep flag)
    """
    df_clean = df.copy()
    df_clean = df_clean.sort_values(by=["ticker", "ts"]).reset_index(drop=True)
    price_cols = ["open", "high", "low", "close", "adj_close"]

    # Recalculate fresh returns on the current cleaned prices 
    # This ensures that if any close or adj close value was interpolated
    # that it is recalculated to be part of the outlier calculations
    prev_c = df_clean.groupby("ticker")["close"].shift(1)
    ret_c = (df_clean["close"] - prev_c) / prev_c

    prev_ac = df_clean.groupby("ticker")["adj_close"].shift(1)
    ret_ac = (df_clean["adj_close"] - prev_ac) / prev_ac

    # Add the columns
    if "outlier_status" not in df_clean.columns:
        df_clean["outlier_status"] = "normal"
    if "evidence_note" not in df_clean.columns:
        df_clean["evidence_note"] = ""
    if "quarantine_status" not in df_clean.columns:
        df_clean["quarantine_status"] = "normal"

    # 1. Evaluate Evidence & Categorize
    for idx in outlier_indices:
        if idx not in df_clean.index:
            continue
        ticker = df_clean.at[idx, "ticker"]

        # Examine which return (adj_close or close) triggered the extreme move
        c_r = ret_c.at[idx]
        ac_r = ret_ac.at[idx]
        use_adj = pd.notna(ac_r) and (pd.isna(c_r) or abs(ac_r) > abs(c_r))
        curr_ret = ac_r if use_adj else c_r

        # Consider the previous and next return (context)
        # To identify if the jump was sustained or if it is likely a one day anomaly
        prev_idx, next_idx = idx - 1, idx + 1
        has_prev = prev_idx in df_clean.index and df_clean.at[prev_idx, "ticker"] == ticker
        has_next = next_idx in df_clean.index and df_clean.at[next_idx, "ticker"] == ticker

        next_ret = (ret_ac.at[next_idx] if use_adj else ret_c.at[next_idx]) if has_next else np.nan

        # Use a multiplicative reversion check to ensure there are no large jumps between days.
        # In escense, this calculates the net return from the day before to the day after as a percentage
        two_day_ret = (1.0 + curr_ret) * (1.0 + next_ret) - 1.0 if (has_next and pd.notna(next_ret)) else np.nan
        is_reverting = (
            has_next
            and pd.notna(next_ret)
            and (np.sign(curr_ret) != np.sign(next_ret)) # is reverting would be a jump in opposite direction
            and (abs(two_day_ret) < 0.15) # Confirm that the change over the 2 day period is net 15% such that
                                          # the jump is an isolated event
        )

        # Also check if adj_close jumped massively while raw close stayed completely normal
        # Important since the summary data explicitly showed many missing values and errors in the adj_close 
        # column, so it is likely that some of that data is misleading. 
        is_adj_corruption = (
            use_adj and pd.notna(c_r) and abs(ac_r) > 0.50 and abs(c_r) < 0.15
        )

        if not has_prev or not has_next:
            df_clean.at[idx, "outlier_status"] = "quarantined"
            df_clean.at[idx, "quarantine_status"] = "outlier_boundary_quarantined"
            df_clean.at[idx, "evidence_note"] = (
                "Boundary row: Insufficient neighbor context to confirm error vs event."
            )
        elif is_reverting or is_adj_corruption:
            df_clean.loc[idx, price_cols] = np.nan
            # Also wipe the return on next_idx so the back-side of a 1-day spike isn't misclassified
            df_clean.at[idx, "outlier_status"] = "confirmed_error_interpolated"
            df_clean.at[idx, "evidence_note"] = (
                f"Single-day spike/corrupted adj_close detected (Return: {curr_ret:.2%}, Next Return: {next_ret:.2%})."
            )
        else:
            df_clean.at[idx, "outlier_status"] = "unverified_extreme_movement"
            df_clean.at[idx, "evidence_note"] = (
                f"Sustained price shift (Return: {curr_ret:.2%}); split/news unconfirmed."
            )

    for col in price_cols:
        df_clean[col] = df_clean.groupby("ticker")[col].transform(
            lambda g: g.interpolate(method="linear", limit_area="inside")
        )

    # Drop temporary helper columns if present
    df_clean = df_clean.drop(columns=["prev_adj_close", "adj_return"], errors="ignore")
    return df_clean

In [20]:
# Clean the data
df_cleaned = fix_impossible_prices_hybrid(df)
df_cleaned = handle_missing_values(df_cleaned)
outlier_indices = df_cleaned[df_cleaned["return_outlier_flag"]].index
df_cleaned = process_flagged_outliers(df_cleaned, outlier_indices)


print("\nQuarantine Status Breakdown:")
display(df_cleaned['quarantine_status'].value_counts())
print(f"Final cleaned dataset shape: {df_cleaned.shape}")


Quarantine Status Breakdown:


quarantine_status
normal                          368248
unresolved_adj_close                 2
outlier_boundary_quarantined         1
Name: count, dtype: int64

Final cleaned dataset shape: (368251, 18)


In [21]:
# Display the first 10 rows of df_cleaned 
# to ensure that the flags are still there and no other data was changed
df_cleaned.head(10)

,ts,open,high,low,close,adj_close,volume,ticker,prev_close,return,return_outlier_flag,impossible_bound_flag,invalid_price_flag,ohlc_inconsistent_flag,zero_volume_flag,quarantine_status,outlier_status,evidence_note
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,False,False,False,False,False,normal,normal,
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,False,False,False,False,False,normal,normal,
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,False,False,False,False,False,normal,normal,
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,False,False,False,False,False,normal,normal,
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,False,False,False,False,False,normal,normal,
5,1980-12-19,0.12612,0.12612,0.12612,0.12612,0.097220,48630447,AAPL,0.11886,0.061080,False,False,False,False,False,normal,normal,
6,1980-12-22,0.13225,0.13225,0.13225,0.13225,0.101950,37363236,AAPL,0.12612,0.048605,False,False,False,False,False,normal,normal,
7,1980-12-23,0.13784,0.13784,0.13784,0.13784,0.106250,46950444,AAPL,0.13225,0.042268,False,False,False,False,False,normal,normal,
8,1980-12-24,0.14509,0.14509,0.14509,0.14509,0.111850,48003248,AAPL,0.13784,0.052597,False,False,False,False,False,normal,normal,
9,1980-12-26,0.15848,0.15848,0.15848,0.15848,0.122170,55574451,AAPL,0.14509,0.092288,True,False,False,False,False,normal,unverified_extreme_movement,Sustained price shift (Return: 9.23%); split/n...


In [22]:
# Verification and Statistics Comparison
post_repair_issues = []

for idx, row in df_cleaned.iterrows():
  # Check for impossible bounds
  if row['high'] < row['low']:
    post_repair_issues.append({
        'ticker': row['ticker'],
        'issue_type': 'Impossible Bound',
        'issue_date': row['ts'],
        'severity': 'high',
    })

  # Check for invalid, missing, or non-positive prices
  for col in ['open', 'high', 'low', 'close', 'adj_close']:
    if pd.isna(row[col]) or row[col] <= 0 or not np.isfinite(row[col]):
      post_repair_issues.append({
          'ticker': row['ticker'],
          'issue_type': 'Invalid Price',
          'issue_date': row['ts'],
          'severity': 'high',
      })

  # Check for zero volume days
  if row['volume'] == 0:
    post_repair_issues.append({
        'ticker': row['ticker'],
        'issue_type': 'Zero Volume',
        'issue_date': row['ts'],
        'severity': 'low',
    })

dq_report_after = pd.DataFrame(post_repair_issues)

# --- SIDE-BY-SIDE SUMMARY COMPARISON ---
print("==========================================")
print(f"Pre-repair baseline: {len(dq_report_before)} issues")
print("==========================================")
display(dq_report_before['issue_type'].value_counts())

print("\n==========================================")
print(f"Post-repair baseline: {len(dq_report_after)} issues")
print("==========================================")
if not dq_report_after.empty:
  display(dq_report_after['issue_type'].value_counts())
else:
  print("Verification Passed: 0 issues remaining!")

# Note: The post repair issues are the zero volume issues that we chose not to address
# according the instructions

Pre-repair baseline: 45984 issues


issue_type
Missing Value (NaN)        19855
Missing Trading Day        13602
Return Outlier (>3σ)       12442
Zero Volume                   75
Invalid Price                  5
Inconsistent OHLC Range        5
Name: count, dtype: int64


Post-repair baseline: 77 issues


issue_type
Zero Volume      75
Invalid Price     2
Name: count, dtype: int64

In [23]:
conn = sqlite3.connect("data.db")
# Write df_cleaned to a NEW table 'ohlc_hw_cleaned'
df_cleaned.to_sql(
    name="ohlc_hw_cleaned",
    con=conn,
    if_exists="replace",  # Overwrites 'ohlc_hw_cleaned' if re-run, leaves 'ohlc_hw' alone
    index=False,  # Prevents saving pandas index as an extra column
)

# Save changes and close connection
conn.commit()
conn.close()

# Re-connect to verify database tables
conn = sqlite3.connect("data.db")

# Query database schema for available tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)
print("Tables currently in data/data.db:")
print(tables)

# Always close connections when finished
conn.close()

Tables currently in data/data.db:
              name
0             ohlc
1          ohlc_hw
2      market_caps
3  ohlc_hw_cleaned
